# jdsl + Gemma residual experiment in Colab

This notebook is the GPU path for Gemma experiments. It installs the jdsl harness branch in Colab, loads a Gemma model from Hugging Face, imports the compiled `.jdsl` package and host tools, and then runs a small residual-decision playground where Gemma returns only `selected_index` and jdsl resolves the exact downstream order ID.

Use `Runtime > Change runtime type > GPU` before running the model cells. The current target model is `google/gemma-4-E2B-it`, whose Hugging Face model card shows the Transformers path with `AutoProcessor` and `AutoModelForMultimodalLM`: https://huggingface.co/google/gemma-4-E2B-it


## 1. Check the Colab GPU

The repository does not require Gemma to run locally. This notebook is intentionally a manual GPU integration artifact.

In [ ]:
import os
import platform
import sys

try:
    import torch
    print('python:', sys.version.split()[0])
    print('platform:', platform.platform())
    print('cuda available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('gpu:', torch.cuda.get_device_name(0))
        print('capability:', torch.cuda.get_device_capability(0))
    else:
        print('Switch Colab to a GPU runtime before loading Gemma.')
except Exception as exc:
    print('torch check failed:', type(exc).__name__, exc)


## 2. Install jdsl and Hugging Face dependencies

This clones the harness branch so examples and `.jdsl` artifacts are present in the runtime filesystem. Change `REPO_URL` or `BRANCH` if you are testing a fork.

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = os.environ.get('JDSL_REPO_URL', 'https://github.com/Cantor-Industries/jdsl-py.git')
BRANCH = os.environ.get('JDSL_BRANCH', 'harness')
REPO_DIR = Path('/content/jdsl-py')

if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)

print('repo:', REPO_DIR)
print('commit:', subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
%pip install -q -e "/content/jdsl-py[harness]"
%pip install -q -U transformers accelerate huggingface_hub safetensors
%cd /content/jdsl-py


## 3. Load a compiled package and host tools

The included `retail.jdsl` is an RDB 0 package, so it is a package/tool-binding sanity check rather than a Gemma transfer proof. It confirms Colab can import jdsl, verify a package, bind trusted host tools, and execute deterministic behavior.

In [ ]:
import importlib.util
from pathlib import Path

from jdsl.package import load_package

def load_tools(path: str):
    spec = importlib.util.spec_from_file_location('retail_tools', path)
    mod = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(mod)
    return mod.TOOLS

PKG = Path('examples/harness/retail.jdsl')
TOOLS = load_tools('examples/harness/retail_tools.py')
pkg = load_package(PKG)

print(f'package: {pkg.manifest.name} v{pkg.manifest.version}')
print('required capabilities:', pkg.manifest.required_capabilities)
print('permissions:', pkg.permissions())
print('verification:', pkg.manifest.verification)


In [ ]:
root = pkg.as_root(TOOLS)

for email in ['ada@example.com', 'bo@example.com', 'cass@example.com']:
    ctx = root.run(email=email)
    order = ctx.blackboard['mcp_retail_get_order_out_3']
    print(f'{email:20} -> {order}')


## 4. Authenticate to Hugging Face

If the selected Gemma repository requires accepting terms or an access token, accept the terms on Hugging Face and add `HF_TOKEN` as a Colab secret. The code also respects an `HF_TOKEN` environment variable.

In [ ]:
HF_TOKEN = os.environ.get('HF_TOKEN')

try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or userdata.get('HF_TOKEN')
except Exception:
    pass

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print('Hugging Face token loaded.')
else:
    print('No HF_TOKEN found. Public models may still load; gated models will fail until a token is provided.')


## 5. Load Gemma behind the jdsl model interface

jdsl residual `predict` leaves call `model.generate(system=..., messages=..., model_id=...)`. This adapter keeps Hugging Face-specific code outside the core package.

In [ ]:
import re
import time
import torch
from transformers import AutoModelForMultimodalLM, AutoProcessor

MODEL_ID = os.environ.get('GEMMA_MODEL_ID', 'google/gemma-4-E2B-it')

class GemmaGenerateModel:
    def __init__(self, model_id: str = MODEL_ID, token: str | None = HF_TOKEN):
        if not torch.cuda.is_available():
            raise RuntimeError('A Colab GPU runtime is required for this Gemma experiment.')
        major, _minor = torch.cuda.get_device_capability(0)
        dtype = torch.bfloat16 if major >= 8 else torch.float16
        self.model_id = model_id
        self.processor = AutoProcessor.from_pretrained(model_id, token=token)
        self.model = AutoModelForMultimodalLM.from_pretrained(
            model_id,
            token=token,
            torch_dtype=dtype,
            device_map='auto',
            low_cpu_mem_usage=True,
        )
        self.model.eval()

    def generate(self, *, system: str, messages: list[dict], model_id: str | None = None) -> str:
        chat = ([{'role': 'system', 'content': system}] if system else []) + list(messages)
        inputs = self.processor.apply_chat_template(
            chat,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors='pt',
        )
        device = next(self.model.parameters()).device
        inputs = {k: (v.to(device) if hasattr(v, 'to') else v) for k, v in inputs.items()}
        prompt_len = inputs['input_ids'].shape[-1]
        started = time.perf_counter()
        with torch.inference_mode():
            out = self.model.generate(
                **inputs,
                max_new_tokens=32,
                do_sample=False,
                pad_token_id=getattr(self.processor.tokenizer, 'eos_token_id', None),
            )
        self.last_elapsed_ms = (time.perf_counter() - started) * 1000
        generated = out[0, prompt_len:]
        return self.processor.decode(generated, skip_special_tokens=True).strip()

class FirstIntegerOutput:
    """Small demo wrapper: return the first integer if Gemma adds extra words."""
    def __init__(self, inner):
        self.inner = inner
        self.model_id = getattr(inner, 'model_id', None)
        self.last_raw = None

    def generate(self, **kwargs) -> str:
        raw = self.inner.generate(**kwargs)
        self.last_raw = raw
        match = re.search(r'-?\d+', raw)
        return match.group(0) if match else raw

gemma_raw = GemmaGenerateModel()
gemma = FirstIntegerOutput(gemma_raw)
print('loaded:', gemma_raw.model_id)


## 6. Probe the residual decision directly

This is the exact kind of local decision an RDB > 0 package should leave to the small model. The model chooses an index; jdsl should own everything around it.

In [ ]:
orders = [
    {'id': '#W991', 'summary': 'red shirt', 'status': 'shipped'},
    {'id': '#W2378156', 'summary': 'blue shoes', 'status': 'pending'},
    {'id': '#W420', 'summary': 'black bag', 'status': 'delivered'},
]

prompt = '\n'.join([
    'request: Show me the order with the blue shoes.',
    'orders:',
    '0: red shirt',
    '1: blue shoes',
    '2: black bag',
    'Return only the integer index.',
])

answer = gemma.generate(
    system='Choose the order the customer refers to. Return only one integer.',
    messages=[{'role': 'user', 'content': prompt}],
)
idx = int(answer)
print('raw Gemma output:', repr(gemma.last_raw))
print('selected_index:', idx)
print('resolved order id:', orders[idx]['id'])


## 7. Run a jdsl residual leaf with Gemma

This builds the runtime shape expected from a compiled RDB > 0 package: `predict(request, orders -> selected_index)`, guard that the dynamic ref exists, then call a deterministic host tool with `orders[$selected_index].id`. The model never regenerates the opaque order ID.

In [ ]:
from jdsl import act, guard, ref, root, seq, store, tool
from jdsl.tree import Predict

@tool
def get_order(order_id: str) -> dict:
    for order in orders:
        if order['id'] == order_id:
            return dict(order)
    raise ValueError(f'no order {order_id!r}')

selector = root(
    'gemma residual order selector',
    system='Choose the order the customer refers to. Return only the integer index.',
).do(seq(
    Predict(
        inputs=('request', 'orders'),
        outputs=('selected_index',),
        instructions='Pick the order requested by the customer. Return only selected_index as an integer.',
        output_schemas={'selected_index': {'type': 'integer', 'minimum': 0}},
    ),
    guard({'exists': {'ref': 'orders[$selected_index].id'}}),
    store(act(get_order, order_id=ref('orders[$selected_index].id')), 'order'),
))

ctx = selector.run(
    model=gemma,
    request='Show me the order with the blue shoes.',
    orders=orders,
)

print('raw Gemma output:', repr(gemma.last_raw))
print('selected_index:', ctx.blackboard['selected_index'])
print('deterministically fetched order:', ctx.blackboard['order'])
print('Gemma latency ms:', round(gemma_raw.last_elapsed_ms, 1))


## 8. What to record from a real package run

For a real RDB > 0 package generated from OpenCode/frontier traces, record the commit SHA, package digest, model ID and revision, Colab GPU type, prompt/output text for each residual leaf, latency, and whether the downstream deterministic tool received the exact intended ID. Do not claim small-model behavior transfer from the RDB 0 package above; it is only a package-binding sanity check.